# 07 · OpenStreetMap downloads (Overpass API)

**Source:** OpenStreetMap contributors, via the Overpass API (`overpass-api.de`). **Licence:** ODbL 1.0 —
cite "© OpenStreetMap contributors".

Three Kerala queries were used: all `amenity=school` (2026-10-28), health (`amenity` hospital/clinic/doctors
+ `healthcare=*`, 2026-11-02), and a wider "school-like" query (schools, colleges, kindergartens,
school buildings and school-like names, 2026-11-04). The files used in the project are the ones with those
dates. OSM changes every day, so running the download again saves a **new file with today's date** and
never overwrites the originals.

## Imports and the three queries

In [1]:
# Project folder. Laptop: open the notebook from "PROJECTS IPYNB" (or the project root).
# Google Colab: upload the project folder to Google Drive ("My Drive") and set its name below.
from pathlib import Path
import sys
PROJECT_FOLDER = "Distance-Capacity-and-Inequality-in-Access-to-Government-and-Aided-Schools"  # Colab only: folder name inside My Drive
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
else:
    PROJECT = Path.cwd().parent if Path.cwd().name in ("notebooks", "PROJECTS IPYNB") else Path.cwd()
assert (PROJECT / "scripts").is_dir(), f"project folder not found: {PROJECT}"
sys.path.insert(0, str(PROJECT / "scripts"))
import datetime as dt
import json
import time

import requests

OSM_DIR = PROJECT / "data" / "raw" / "osm"
OVERPASS = "https://overpass-api.de/api/interpreter"
# The three queries used in the project, exactly as sent on the dates in the file names
QUERIES = {
    "osm_kerala_schools": (  # 2026-10-28
        '[out:json][timeout:300];area["ISO3166-2"="IN-KL"][admin_level=4]->.k;'
        'nwr["amenity"="school"](area.k);out center tags;'),
    "osm_kerala_health": (  # 2026-11-02
        '[out:json][timeout:180];area["ISO3166-2"="IN-KL"]->.k;'
        '(nwr["amenity"~"^(hospital|clinic|doctors)$"](area.k);nwr["healthcare"](area.k););out center tags;'),
    "osm_kerala_schoolish": (OSM_DIR / "osm_kerala_schoolish_2026-11-04.overpassql").read_text(encoding="utf-8"),  # 2026-11-04
}
ORIGINAL = {"osm_kerala_schools": "2026-10-28", "osm_kerala_health": "2026-11-02", "osm_kerala_schoolish": "2026-11-04"}

In [ ]:
# Colab only: install packages that Colab does not have by default (does nothing on the laptop)
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz", "pymupdf", "pyogrio"], check=True)

### `download()`

In [2]:
def download(name, query, date=None):
    """Send one Overpass query and save the answer as <name>_<date>.json (never overwrites a file)."""
    date = date or dt.date.today().isoformat()
    f = OSM_DIR / f"{name}_{date}.json"
    if f.exists():
        print("already saved:", f.name)
        return f
    for attempt in range(3):
        r = requests.post(OVERPASS, data={"data": query}, timeout=600,
                          headers={"User-Agent": "msc-spatial-inequality-study/0.1 (academic project)"})
        if r.ok:
            f.write_text(r.text, encoding="utf-8")
            (OSM_DIR / f"{name}_{date}.overpassql").write_text(query, encoding="utf-8")
            print("saved:", f.name, len(r.json()["elements"]), "elements")
            return f
        print("retry", attempt + 1, r.status_code)
        time.sleep(30)
    r.raise_for_status()

## Run

Downloads today's OSM data (about 3 × 10 MB). Skip this cell to use only the original files.

In [3]:
for name, q in QUERIES.items():
    download(name, q)
    time.sleep(10)  # be polite to the public Overpass server

saved: osm_kerala_schools_2026-11-07.json 15129 elements


saved: osm_kerala_health_2026-11-07.json 13904 elements


retry 1 502


saved: osm_kerala_schoolish_2026-11-07.json 52404 elements


## Check the original files used in the project

In [4]:
for name, date in ORIGINAL.items():
    f = OSM_DIR / f"{name}_{date}.json"
    print(f"{f.name:40s}", len(json.loads(f.read_text(encoding="utf-8"))["elements"]) if f.exists() else "missing", "elements")

osm_kerala_schools_2026-10-28.json       15123 elements
osm_kerala_health_2026-11-02.json        13885 elements


osm_kerala_schoolish_2026-11-04.json     52397 elements
